# LucaOne: frozen representation comparisons

How well do LucaOne's frozen representations predict mRNA stability compared with BioLangFusion's original encoders and features calculated directly from the sequence? This Stage 1 notebook keeps checkpoint feasibility, pooled embedding preparation and representation comparisons together. It retains the earlier feasibility outputs and the separately executed stability comparison; moving them here does not establish a new execution.

LucaOne's checkpoint was pretrained on nucleotide and protein sequences. Here, its weights remain frozen: we encode each coding sequence and its translated protein separately, average their token representations, and fit ridge regression to either vector or their concatenation. The same ridge procedure reads the existing BioLangFusion encoder vectors. This compares representation sources, not locally trained token-level fusion architectures.

The focused constituent comparison asks whether concatenating LucaOne's gene and protein vectors improves on either vector alone. It reads existing caches and uses validation-selected ridge penalties, leaving the earlier tables intact. Its separate switch does not enable encoder extraction. The [frozen GTEx fusion reference](../stage3/gtex_frozen_fusion_reference.ipynb) now owns the separate architecture, pilot and repeat-seed investigation.


The [Stage 1 guide](README.md) connects this comparison to the other representation diagnostics. [BioLangFusion token extraction](../stage3/blf_token_embeddings.ipynb) supplies the original encoder caches reused below; completing encoder swaps is not a dependency. The [research plan](../../docs/Research_Project_Plan_Gantt.xlsx), `Gantt` and `Details` task 41, records the LucaOne aim.

**Execution and retained evidence.** `READ_SAVED_RESULTS = True` preserves the saved-table reading route and also skips the relocated feasibility calculations and Hub query. With it set to `False`, the existing feasibility, timing, extraction and historical probe paths can run. The optional constituent follow-up has its own switch. Existing outputs, counts and cell metadata remain attached to their original cells, including setup/diagnostic output from the earlier mixed notebooks; they are historical evidence, not outputs from this arrangement.


---

## Setup

Prepare the environment using the root README's [notebook dependencies](../../README.md#notebook-dependencies) and select that environment as the notebook kernel. Package installation belongs in that setup step.

### Imports


In [1]:
# Files, project paths, serialized checkpoint names and elapsed-time measurements.
import json
import os
import sys
import time
from pathlib import Path

# Enable CPU fallback for unsupported Apple GPU operations before importing torch.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays and result tables.
import numpy as np
import pandas as pd
from IPython.display import display

# Encoder inference, device selection and numerical precision.
import torch

# Ridge prediction, feature scaling and their shared fitting pipeline.
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Display width is measured in characters, not sequence positions.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print(
    "torch", torch.__version__,
    "| CUDA:", torch.cuda.is_available(),
    "| Apple GPU:", torch.backends.mps.is_available(),
)


torch 2.10.0+cu126 | CUDA: True | Apple GPU: False


### Project paths and shared code

Locate the project's `Code` directory before importing the shared helpers. Input and result locations below are anchored to this directory so they refer to the same files when the kernel starts elsewhere inside the repository.


In [ ]:
# Find Code/shared_code from the repository root or a folder beneath it.
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    raise FileNotFoundError(
        "Cannot locate Code/shared_code. Start this notebook from within the repository."
    )

# Make shared_code importable without changing the kernel's working folder.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Dataset preparation, frozen encoders and sequence-derived features.
from shared_code import datasets, encoders, lucaone, matched_probes, saved_results, sequences

# Fusion configurations, GTEx inputs, prediction scores and saved results.
from shared_code.fusion import blf, isoformer, isoformer_results, run_identity, swap_results, train
from shared_code.fusion import token_embeddings as tokens


### Settings, file locations and hardware

**Read saved results by default.** With `READ_SAVED_RESULTS = True`, Run All reads the existing LucaOne CSV summaries and Phase A/IsoFormer result metadata. It skips source-data inspection, timing, model downloads, cache opening, fitting, bootstrap calculations and file writes. Missing or invalid summaries are reported without starting their producer; independent result sections can still display.

The CSV readers check columns, finite numbers, duplicate entries and expected coverage. These historical summaries do not fully record input identity, actual seed IDs or producer settings, so passing those checks does not establish compatibility with changed experiment settings. The explanations and retained outputs below describe the existing calculation, not a fresh run.

The selected benchmark is mRNA stability, and `CHECKPOINT` chooses the jointly pretrained LucaOne checkpoint. “Jointly pretrained” describes how the checkpoint learned; it does not mean that this notebook feeds the nucleotide and protein sequences into it together.

The device preference is NVIDIA CUDA, then Apple MPS, then CPU. LucaOne uses float16 on CUDA and its default float32 elsewhere. Half precision reduces weight storage, but weight bytes alone do not establish peak RAM or GPU memory: activations and other allocations also matter. The timing section reports the loaded model's parameter count; the earlier 1.8B family description is not the parameter count measured for every local load.

Set `READ_SAVED_RESULTS = False` only when intending to execute the original calculation path. In that mode, `RUN_LUCAONE` controls full embedding extraction or cache loading, but the timing trial runs independently of it; `RUN_ISOFORMER_SEEDS` controls GTEx training in Part 4. That path may download, fit and replace summary files as before; it is not a compute-only-missing mode.


In [ ]:
# Read existing summaries by default; False enables the original computation path.
READ_SAVED_RESULTS = True

# Selected stability comparison and LucaOne checkpoint.
BENCHMARK = "mrna_stability"
CHECKPOINT = "default"

# These execution choices apply only when READ_SAVED_RESULTS is False.
RUN_LUCAONE = True

# Independent CPU follow-up: read existing caches, fit probes and save a new analysis.
RUN_LUCAONE_COMPARISON = False
if RUN_LUCAONE_COMPARISON and not READ_SAVED_RESULTS:
    raise ValueError("Keep READ_SAVED_RESULTS=True for the cache-only LucaOne comparison.")

# Bootstrap resamples the fixed test predictions; it does not refit the probe.
BOOTSTRAP_DRAWS = 4000
PROBE_SEEDS = (0, 1, 2)  # Retained legacy setting; no code consumes it here.

# Reuse the existing Stage 3 embeddings and Stage 4 LucaOne result locations.
STAGE3_DIR = CODE_DIR / "stage3"
STAGE4_DIR = CODE_DIR / "stage4"
CACHE_ROOT = STAGE3_DIR / "embeddings"
LUCAONE_DIR = CACHE_ROOT / BENCHMARK / "lucaone"
RESULTS_DIR = STAGE4_DIR / "results" / "lucaone"

# Keep the prior ridge penalty grid; select on validation MSE in this follow-up.
LUCAONE_ALPHAS = tuple(np.logspace(-3, 5, 20))
LUCAONE_COMPARISON_PATH = RESULTS_DIR / "complementarity_v1" / f"{BENCHMARK}.json"


In [12]:
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
DTYPE = torch.float16 if DEVICE == "cuda" else None
print(f"device {DEVICE}, LucaOne dtype {DTYPE or 'float32'}")
print(f"checkpoints published: {json.dumps(lucaone.CHECKPOINTS, indent=2)}")


device cuda, LucaOne dtype torch.float16
checkpoints published: {
  "default": "LucaGroup/LucaOne-default-step36M",
  "gene": "LucaGroup/LucaOne-gene-step36.8M",
  "prot": "LucaGroup/LucaOne-prot-step38.2M"
}


---

## Check the selected checkpoint scope

LucaOne provides a jointly pretrained nucleic-acid/protein representation family, whereas this project's fusion experiments combine representations from selected encoders downstream. The [LucaOne source record](../../docs/sources.md#he-et-al-lucaone) explains that reference role and the separate-input embedding interface.

The following check retains a **1,280-token whole-input limit** and character-level length arithmetic. A nucleotide or amino-acid character contributes one token, with two framing tokens added in the calculation. These are the selected feasibility assumptions, not a tokenizer execution or a claim about every release and long-sequence strategy.

For GTEx, the table examines the genomic DNA window, transcript RNA, untranslated regions, coding sequence and protein. For the filtered BLF benchmarks, protein length is estimated as the number of complete triplets. The “both in one sequence” column is a **hypothetical combined-input budget**, adding nucleotide length, estimated protein length and three special tokens. It does not establish a supported LucaOne joint-input API. The printed 957-nucleotide limit refers to complete-codon lengths under that approximation.


In [ ]:
# Selected whole-input scope; the check does not load a LucaOne tokenizer or model.
LUCAONE_LIMIT = 1280
LUCAONE_SPECIAL_TOKENS = 2  # [CLS] and [SEP] in the retained length arithmetic


In [ ]:
if READ_SAVED_RESULTS:
    print('Saved-results mode: historical GTEx length diagnostic skipped.')
else:
    gtex = pd.read_csv(datasets.DEFAULT_DATA_DIR / isoformer.GTEX_PILOT)
    # Casting to strings preserves the old check; it does not validate missing fields or alphabets.
    gtex_length_rows = []
    for column in ("DNA", "RNA", "5UTR", "CDS", "3UTR", "Protein"):
        sequence_lengths = gtex[column].astype(str).str.strip().str.len()
        gtex_length_rows.append(
            {
                "GTEx column": column,
                "min": sequence_lengths.min(),
                "median": int(sequence_lengths.median()),
                "max": sequence_lengths.max(),
                "fits LucaOne": f"{(sequence_lengths + LUCAONE_SPECIAL_TOKENS <= LUCAONE_LIMIT).mean() * 100:.1f}%",
            }
        )
    display(pd.DataFrame(gtex_length_rows).set_index("GTEx column"))


In [2]:
if READ_SAVED_RESULTS:
    print('Saved-results mode: historical BLF length diagnostic skipped.')
else:
    # For the filtered BLF rows, estimate protein length as complete triplets: length // 3.
    benchmark_length_rows = []
    for benchmark in datasets.BLF_BENCHMARKS:
        sequence_lengths = datasets.blf_benchmark_rows(benchmark)["length"]
        benchmark_length_rows.append(
            {
                "benchmark": benchmark,
                "rows": len(sequence_lengths),
                "max nt": sequence_lengths.max(),
                "nucleotides alone": f"{(sequence_lengths + LUCAONE_SPECIAL_TOKENS <= LUCAONE_LIMIT).mean() * 100:.1f}%",
                "protein alone": f"{(sequence_lengths // 3 + LUCAONE_SPECIAL_TOKENS <= LUCAONE_LIMIT).mean() * 100:.1f}%",
                "both in one sequence": f"{(sequence_lengths + sequence_lengths // 3 + LUCAONE_SPECIAL_TOKENS + 1 <= LUCAONE_LIMIT).mean() * 100:.1f}%",
            }
        )
    display(pd.DataFrame(benchmark_length_rows).set_index("benchmark"))
    print(
        f"The hypothetical combined-input budget uses 4L/3 + {LUCAONE_SPECIAL_TOKENS + 1} <= {LUCAONE_LIMIT}, "
        f"so the largest complete-codon sequence under that approximation is {(LUCAONE_LIMIT - LUCAONE_SPECIAL_TOKENS - 1) * 3 // 4} nt."
    )


,min,median,max,fits LucaOne
GTEx column,,,,
DNA,6000,6000,6000,0.0%
RNA,78,1723,21116,39.9%
5UTR,1,152,3318,99.6%
CDS,3,760,16080,68.1%
3UTR,1,390,17986,76.6%
Protein,1,253,5360,96.8%


,rows,max nt,nucleotides alone,protein alone,both in one sequence
benchmark,,,,,
fungal_expression,3138,999,100.0%,100.0%,93.7%
ecoli_proteins,4450,999,100.0%,100.0%,96.4%
mrna_stability,23929,999,100.0%,100.0%,93.4%
cov_vac_degradation,2400,81,100.0%,100.0%,100.0%


The joint mode needs 4L/3 + 3 <= 1280, so it caps the coding sequence at 957 nt.


The retained GTEx table shows why the full 6,000-nucleotide DNA window cannot fit this selected whole-input rule. It also shows that many full transcripts exceed the limit. Cropping, windowing or changing the admitted transcript population would define a different input comparison; the table does not rule out every LucaOne GTEx approach.

The BLF tables ask a narrower question: how much of each already length-filtered benchmark fits the arithmetic budget. They do not establish predictive quality, compute cost or biological validity of the estimated protein stream. String lengths also do not validate missing values or sequence alphabets.

### Check released model metadata

In computation mode, the next cell queries repository names and file listings, without loading a tokenizer, downloading weights or constructing a model. The saved response establishes what that query returned at its execution, not current availability or successful loading. A metadata error can reflect access or network failure rather than an absent model.


In [3]:
if READ_SAVED_RESULTS:
    print('Saved-results mode: Hub metadata query skipped.')
else:
    # Query model/file metadata over the network; no weights or model loader are used.
    model_metadata_rows = []
    try:
        from huggingface_hub import HfApi

        hub_api = HfApi()
        # Retain the search and fallback order; repository names do not prove loader compatibility.
        published_models = [
            model_summary.id for model_summary in hub_api.list_models(author="LucaGroup")
        ]
        print(
            f"LucaGroup publishes {len(published_models)} models:", ", ".join(sorted(published_models))
        )
        candidate_models = (
            sorted(
                model_summary
                for model_summary in published_models
                if "lucaone" in model_summary.lower()
            )
            or published_models[:4]
        )
        for model_id in candidate_models:
            try:
                model_info = hub_api.model_info(model_id, files_metadata=False)
                model_files = (
                    [file_record.rfilename for file_record in model_info.siblings]
                    if model_info.siblings
                    else []
                )
                model_metadata_rows.append(
                    {
                        "repo": model_id,
                        "exists": "yes",
                        "config.json": "yes" if "config.json" in model_files else "no",
                        "files": len(model_files),
                        "weights": ", ".join(
                            filename
                            for filename in model_files
                            if filename.endswith((".bin", ".safetensors", ".pt"))
                        )[:60],
                    }
                )
            except Exception as error:
                model_metadata_rows.append(
                    {
                        "repo": model_id,
                        "exists": f"unverified ({type(error).__name__})",
                        "config.json": "",
                        "files": 0,
                        "weights": "",
                    }
                )
    except ImportError:
        print("huggingface_hub is not installed; skipping the probe")

    if model_metadata_rows:
        display(pd.DataFrame(model_metadata_rows).set_index("repo"))
        print(
            "Listed configuration and weight files are availability evidence, not a successful load.\n"
            "Loader compatibility must be checked separately; metadata errors do not prove absence."
        )


LucaGroup publishes 13 models: LucaGroup/LucaCell-v1.0-step6.4M, LucaGroup/LucaOne-LucaCell-step60M, LucaGroup/LucaOne-default-step17.6M, LucaGroup/LucaOne-default-step36M, LucaGroup/LucaOne-default-step5.6M, LucaGroup/LucaOne-gene-step36.8M, LucaGroup/LucaOne-mask-step36M, LucaGroup/LucaOne-prot-step38.2M, LucaGroup/LucaVirus-default-step3.8M, LucaGroup/LucaVirus-gene-step3.8M, LucaGroup/LucaVirus-mask-step1.4M, LucaGroup/LucaVirus-mask-step3.8M, LucaGroup/LucaVirus-prot-step3.8M


,exists,config.json,files,weights
repo,,,,
LucaGroup/LucaOne-LucaCell-step60M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-default-step17.6M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-default-step36M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-default-step5.6M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-gene-step36.8M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-mask-step36M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-prot-step38.2M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."


A repo without config.json will not load through transformers AutoModel and
needs LucaOne's own loader from its GitHub release.


The feasibility check motivates a scoped conclusion: preserve the selected uncropped GTEx input requirement, and consider LucaOne on shorter inputs separately. It does not close every question under Gantt 41. The stability comparison below uses separately encoded LucaOne nucleotide and protein vectors with ridge predictors; that is distinct from the [trained GTEx attention heads](../stage3/gtex_frozen_fusion_reference.ipynb).


---

## Part 1: which inputs fit the selected LucaOne context?

For the stability benchmark, a row supplies a coding sequence and a numerical stability target. LucaOne's gene stream reads the nucleotide sequence; its protein stream reads the translated amino-acid sequence. Separate calls allow each input its own context. A joint input would place both sequences in one context, leaving less room for each.

The coverage helper checks the selected 1,280-token context, including framing tokens. For a complete coding sequence of length `L` nucleotides, a joint input needs `L + L // 3 + 3` tokens: nucleotides, translated residues and framing tokens. The largest complete coding sequence fitting that rule is 957 nucleotides. [The shared LucaOne helper](../shared_code/lucaone.py) implements this arithmetic.

The table also examines GTEx's genomic DNA windows, full transcripts (`RNA`) and coding regions (`CDS`). DNA-window and full-transcript rows report nucleotide-context coverage only; their protein and joint columns are marked “not applicable.” Dividing those nucleotide lengths by three would count noncoding regions as translated residues. The CDS row retains the helper's in-frame translation-length estimate; this is a context-size check, not an extraction or fusion experiment.


In [13]:
if READ_SAVED_RESULTS:
    coverage = []
    print("Saved-results mode: fresh sequence/context inspection skipped; no dataset files opened.")
else:
    coverage = []
    for key in datasets.BLF_BENCHMARKS:
        rows = datasets.blf_benchmark_rows(key)
        coverage.append({
            "benchmark": key,
            **lucaone.coverage(rows["length"].to_numpy()),
        })

    gtex = pd.read_csv(datasets.DEFAULT_DATA_DIR / isoformer.GTEX_PILOT)
    for column in ("DNA", "RNA", "CDS"):
        length = gtex[column].astype(str).str.strip().str.len()
        context = lucaone.coverage(length.to_numpy())
        if column != "CDS":
            # Genomic windows and transcripts include noncoding sequence: division by
            # three cannot infer their protein or joint-input lengths.
            context["protein stream"] = np.nan
            context["joint input"] = np.nan
        coverage.append({"benchmark": f"gtex_pilot/{column}", **context})

    table = pd.DataFrame(coverage).set_index("benchmark")
    display(table.style.format({
        "gene stream": "{:.1%}",
        "protein stream": "{:.1%}",
        "joint input": "{:.1%}",
    }, na_rep="not applicable"))
    print(
        f"The joint mode needs 4L/3 + 3 tokens, so it caps a coding sequence at "
        f"{lucaone.longest_joint_cds()} nt against LucaOne's "
        f"{lucaone.CONTEXT_TOKENS}-token context."
    )


,rows,longest nt,gene stream,protein stream,joint input
benchmark,,,,,
fungal_expression,3138,999,100.0%,100.0%,93.7%
ecoli_proteins,4450,999,100.0%,100.0%,96.4%
mrna_stability,23929,999,100.0%,100.0%,93.4%
cov_vac_degradation,2400,81,100.0%,100.0%,100.0%
gtex_pilot/DNA,2000,6000,0.0%,0.0%,0.0%
gtex_pilot/RNA,2000,21116,39.9%,83.9%,31.6%
gtex_pilot/CDS,2000,16080,68.1%,96.8%,58.0%


The joint mode needs 4L/3 + 3 tokens, so it caps a coding sequence at 957 nt against LucaOne's 1280-token context.


**What this establishes.** The preprocessed BioLangFusion coding sequences fit separately within the selected context; stability is the benchmark chosen for this comparison, not the only BioLangFusion benchmark that fits. Some coding sequences exceed the joint-input limit, so the extraction below encodes nucleotide and protein inputs separately.

GTEx's 6,000-nucleotide DNA windows exceed the selected whole-input context. This notebook therefore omits the LucaOne GTEx comparison. That is a restriction of this comparison, not proof that LucaOne cannot process GTEx under any procedure: the local helper supports windowed pooling, but using it here would change the input treatment. The previously recorded checkpoint configuration has a larger position setting with position embeddings disabled; this comparison still uses the selected 1,280-token rule. The [Stage 4 guide](README.md#lucaone-comparing-jointly-pretrained-representations) retains this scope distinction.


---

## Part 2: timing and extracting LucaOne representations

Load the selected stability rows before preparing their two input streams. The length range below refers to the preprocessed coding sequences actually used by this comparison.


In [ ]:
if READ_SAVED_RESULTS:
    rows = None
    print("Saved-results mode: source dataset loading skipped.")
else:
    rows = datasets.blf_benchmark_rows(BENCHMARK)


### Measure a small inference sample

Saved-results mode skips this trial. In computation mode, this cell loads LucaOne, potentially downloads missing weights into the Hugging Face cache, and runs inference on the first eight gene sequences. It runs regardless of `RUN_LUCAONE`. It does not save sequence embeddings.

The projection multiplies the measured gene-input rate by the number of rows and doubles it for two streams. These first rows are not a representative random timing sample, and protein inputs need not cost the same as gene inputs. Treat the result as a rough estimate for that device and sample, not a completion-time guarantee or a peak-memory measurement.


In [14]:
if READ_SAVED_RESULTS:
    print("Saved-results mode: LucaOne loading, timing and inference skipped.")
else:
    print(
        f"{BENCHMARK}: {len(rows):,} rows, "
        f"{rows['length'].min()} to {rows['length'].max()} nt"
    )

    # The gene tokenizer maps T and U to the same nucleotide token.
    PROBE_ROWS = 8
    sample = rows["sequence"].head(PROBE_ROWS).tolist()
    started = time.perf_counter()
    tokenizer, model = lucaone.load(CHECKPOINT, DEVICE, DTYPE)
    load_seconds = time.perf_counter() - started
    parameters = sum(p.numel() for p in model.parameters())

    started = time.perf_counter()
    for text in sample:
        lucaone.pooled(text, "gene", tokenizer, model, DEVICE)
    per_sequence = (time.perf_counter() - started) / len(sample)

    del model, tokenizer
    encoders.free_memory()

    print(
        f"LucaOne {CHECKPOINT}: {parameters / 1e9:.2f}B parameters, "
        f"loaded in {load_seconds:.0f} s"
    )
    print(f"{per_sequence:.2f} s per sequence on {DEVICE}")
    for streams, label in ((1, "gene stream only"), (2, "gene and protein streams")):
        hours = per_sequence * len(rows) * streams / 3600
        print(
            f"  {label}: {hours:.1f} hours for "
            f"{len(rows) * streams:,} forward passes"
        )


mrna_stability: 23,929 rows, 3 to 999 nt


Loading weights:   0%|          | 0/344 [00:00<?, ?it/s]

[transformers] LucaGPLMModel LOAD REPORT from: LucaGroup/LucaOne-default-step36M
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LucaOne default: 1.57B parameters, loaded in 33 s
0.19 s per sequence on cuda
  gene stream only: 1.3 hours for 23,929 forward passes
  gene and protein streams: 2.5 hours for 47,858 forward passes


### Prepare and embed the two streams

The gene input is the benchmark's coding sequence. The protein input uses `grid_protein`, the translation also used by BioLangFusion's Phase A protein encoder: complete codons are translated and stop symbols become `X`. LucaOne produces token representations, which the local helper averages over content tokens to obtain one vector per input. “Both streams” later means concatenating those two separately obtained vectors.

Each stream has its own cache. Reuse checks include checkpoint name, stream type, context and an ordered-input fingerprint; they do not establish every aspect of the original extraction environment. Progress messages count processed sequences, but the cache is published only after the **whole stream** finishes. An interrupted partial stream must be recomputed; a completed stream can be reused.

In computation mode, the manifest is written for the current invocation, including when matrices are reused. Its device and precision fields should not be treated as proof of the cached matrices' original producing environment.


In [ ]:
if READ_SAVED_RESULTS:
    gene_texts, protein_texts = [], []
else:
    gene_texts = rows["sequence"].tolist()
    protein_texts = [tokens.grid_protein(s) for s in rows["sequence"]]


In [15]:
if READ_SAVED_RESULTS:
    streams = {}
    print("Saved-results mode: embedding caches and manifest writes skipped.")
else:
    LUCAONE_DIR.mkdir(parents=True, exist_ok=True)

    streams = {}
    if RUN_LUCAONE:
        for name, texts in (("gene", gene_texts), ("prot", protein_texts)):
            started = time.perf_counter()
            matrix, reused = lucaone.embed_with_cache(
                texts,
                name,
                LUCAONE_DIR / f"{CHECKPOINT}_{name}.npz",
                device=DEVICE,
                checkpoint=CHECKPOINT,
                dtype=DTYPE,
            )
            streams[name] = matrix
            print(
                f"  {name}: {matrix.shape}, "
                f"{'reused' if reused else f'{(time.perf_counter() - started) / 60:.1f} min'}"
            )

        # This records the invocation even when both matrices came from existing caches.
        lucaone.write_manifest(
            LUCAONE_DIR / "manifest.json",
            benchmark=BENCHMARK,
            checkpoint=lucaone.CHECKPOINTS[CHECKPOINT],
            rows=len(rows),
            device=DEVICE,
            dtype=str(DTYPE),
            context=lucaone.CONTEXT_TOKENS,
        )
    else:
        print("RUN_LUCAONE is off. Read the projection above, then set it to True and rerun.")


Loading weights:   0%|          | 0/344 [00:00<?, ?it/s]

[transformers] LucaGPLMModel LOAD REPORT from: LucaGroup/LucaOne-default-step36M
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    lucaone/gene: 200/23929
    lucaone/gene: 400/23929
    lucaone/gene: 600/23929
    lucaone/gene: 800/23929
    lucaone/gene: 1000/23929
    lucaone/gene: 1200/23929
    lucaone/gene: 1400/23929
    lucaone/gene: 1600/23929
    lucaone/gene: 1800/23929
    lucaone/gene: 2000/23929
    lucaone/gene: 2200/23929
    lucaone/gene: 2400/23929
    lucaone/gene: 2600/23929
    lucaone/gene: 2800/23929
    lucaone/gene: 3000/23929
    lucaone/gene: 3200/23929
    lucaone/gene: 3400/23929
    lucaone/gene: 3600/23929
    lucaone/gene: 3800/23929
    lucaone/gene: 4000/23929
    lucaone/gene: 4200/23929
    lucaone/gene: 4400/23929
    lucaone/gene: 4600/23929
    lucaone/gene: 4800/23929
    lucaone/gene: 5000/23929
    lucaone/gene: 5200/23929
    lucaone/gene: 5400/23929
    lucaone/gene: 5600/23929
    lucaone/gene: 5800/23929
    lucaone/gene: 6000/23929
    lucaone/gene: 6200/23929
    lucaone/gene: 6400/23929
    lucaone/gene: 6600/23929
    lucaone/gene: 6800/23929
    lucaone/gene: 

Loading weights:   0%|          | 0/344 [00:00<?, ?it/s]

[transformers] LucaGPLMModel LOAD REPORT from: LucaGroup/LucaOne-default-step36M
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    lucaone/prot: 200/23929
    lucaone/prot: 400/23929
    lucaone/prot: 600/23929
    lucaone/prot: 800/23929
    lucaone/prot: 1000/23929
    lucaone/prot: 1200/23929
    lucaone/prot: 1400/23929
    lucaone/prot: 1600/23929
    lucaone/prot: 1800/23929
    lucaone/prot: 2000/23929
    lucaone/prot: 2200/23929
    lucaone/prot: 2400/23929
    lucaone/prot: 2600/23929
    lucaone/prot: 2800/23929
    lucaone/prot: 3000/23929
    lucaone/prot: 3200/23929
    lucaone/prot: 3400/23929
    lucaone/prot: 3600/23929
    lucaone/prot: 3800/23929
    lucaone/prot: 4000/23929
    lucaone/prot: 4200/23929
    lucaone/prot: 4400/23929
    lucaone/prot: 4600/23929
    lucaone/prot: 4800/23929
    lucaone/prot: 5000/23929
    lucaone/prot: 5200/23929
    lucaone/prot: 5400/23929
    lucaone/prot: 5600/23929
    lucaone/prot: 5800/23929
    lucaone/prot: 6000/23929
    lucaone/prot: 6200/23929
    lucaone/prot: 6400/23929
    lucaone/prot: 6600/23929
    lucaone/prot: 6800/23929
    lucaone/prot: 

---

## Part 3: compare representations with the same ridge predictor

The representations are now inputs to a smaller fitted predictor. Ridge regression learns a weighted combination of features while penalizing large coefficients. Applying the same predictor family and supplied partitions to each representation makes their predictive performance comparable under this procedure. Checkpoint capacity, vector width and precision still differ, so a score difference does not isolate joint pretraining as its cause. The individual and concatenated vectors address representation usefulness and predictive complementarity under ridge regression; they do not compare alternative learned fusion architectures.

### Prepare partitions and comparison inputs

`Value` is a standardized RNA-stability score: the source aligns the direction of RNA-decay slopes so higher values indicate greater stability, then calculates a z-score separately within each experimental profile. It is not a half-life in hours. The released sequence/value observations come from nine profiles spanning multiple cell contexts and species, but the benchmark table does not retain those profile identifiers. The [target guide](../../docs/methods/data-and-targets.md#mrna-stability-standardized-decay-profiles) traces the measurements, standardization, and consequences for comparing rows. This notebook reads the supplied `Value` directly; its Spearman score evaluates prediction of the stored ordering, not a percentage change in RNA survival.

`Split` identifies the supplied partitions. The predictor fits on training rows and is scored on test rows. The supplied validation rows are unused: `RidgeCV` selects its penalty internally using the training data. The original reason for choosing this procedure over the supplied validation partition is not established.

The BioLangFusion vectors are means of content-token representations from Stage 3's existing caches. Their row order must correspond to the current benchmark rows; opening these caches here is not a fresh verification of every input's identity. The calculated baseline has **87 features**: GC content, GC at the third codon position, log sequence length, 64 codon frequencies and 20 amino-acid frequencies. Its retained table label abbreviates that feature list. [Sequence feature code](../shared_code/sequences.py) defines the calculation.


In [ ]:
if READ_SAVED_RESULTS:
    split = y = None
    train_rows, test_rows = [], []
else:
    # Keep the supplied partitions; do not construct new folds or shuffle row order.
    split = rows["Split"].to_numpy()
    y = rows["Value"].to_numpy(dtype=float)
    train_rows = np.flatnonzero(split == "train")
    test_rows = np.flatnonzero(split == "test")


In [16]:
if READ_SAVED_RESULTS:
    pooled, composition = {}, None
    print("Saved-results mode: token-cache pooling and composition calculation skipped.")
else:
    print(f"published split: {len(train_rows):,} train, {len(test_rows):,} test")

    # Average existing content-token representations into one vector per input row.
    pooled = {}
    for modality, encoder_key in blf.ENCODER_OF_MODALITY.items():
        cache = tokens.summarize(CACHE_ROOT / BENCHMARK / encoder_key)
        pooled[encoders.ENCODERS[encoder_key].label] = tokens.pooled_matrix(cache)
        print(f"  {encoder_key}: {pooled[encoders.ENCODERS[encoder_key].label].shape}")

    # Sequence-derived features include amino-acid composition as well as codon use.
    composition = np.array([
        sequences.composition_features(s) for s in rows["sequence"]
    ])
    print(f"  composition features: {composition.shape}")


published split: 16,742 train, 3,579 test
  ntv2_100m: (23929, 512)
  rnafm: (23929, 640)
  esm2_8m: (23929, 320)
  composition features: (23929, 87)


### Define the predictor and assemble representations

The pipeline learns feature scaling from training rows and then fits `RidgeCV` over the existing penalty grid. A fresh pipeline is constructed for every representation. Test rows are transformed using the fitted scaling and used only to obtain predictions and their Spearman rank correlation with the stability targets. The [linear-probing guide](../../docs/methods/sequences-to-predictions.md#linear-probing) explains why this measures what the selected predictor can recover from frozen representations.

Concatenation joins vectors for the same row along the feature axis. It neither updates the encoders nor introduces token-level cross-attention. Comparing an embedding predictor with the composition predictor is also different from fitting embeddings to residual errors after composition controls, as the mRFP notebook does.


In [ ]:
def probe_on_split(X, y, train_rows, test_rows):
    # Fit scaling and the ridge penalty using training rows, then predict test rows.
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 5, 20))),
    ])
    model.fit(X[train_rows], y[train_rows])
    predicted = model.predict(X[test_rows])
    return predicted, train.score_rows("regression", y[test_rows], predicted)


In [ ]:
if READ_SAVED_RESULTS:
    representations = {}
else:
    representations = {
        "Composition (GC, GC3, codon use, length)": composition,
        **pooled,
        "BLF three encoders concatenated": np.hstack(list(pooled.values())),
    }
    if streams:
        representations["LucaOne, gene stream"] = streams["gene"]
        representations["LucaOne, protein stream"] = streams["prot"]
        representations["LucaOne, both streams"] = np.hstack([
            streams["gene"], streams["prot"],
        ])


Saved-results mode reads the existing probe-score CSV and checks that it contains the expected representations; it does not reconstruct predictions. In computation mode, fit one predictor per representation and retain its test predictions for the paired comparison afterward. In the table, `width` counts input features per row; `probe Spearman` measures agreement between predicted and observed stability rankings. Neither column measures training speed or biological mechanism.


In [17]:
if READ_SAVED_RESULTS:
    # The CSV retains aggregate scores, not the predictions used to obtain them.
    scores, predictions, summary = {}, {}, None
    expected_representations = [
        "Composition (GC, GC3, codon use, length)",
        *(encoders.ENCODERS[key].label for key in blf.ENCODER_OF_MODALITY.values()),
        "BLF three encoders concatenated",
        "LucaOne, gene stream", "LucaOne, protein stream", "LucaOne, both streams",
    ]
    saved_rows = saved_results.read_historical_csv(
        RESULTS_DIR / f"{BENCHMARK}_probe_scores.csv",
        required=("", "probe Spearman", "width"), key=("",),
        numeric=("probe Spearman",), integers=("width",),
        expected_keys=[(name,) for name in expected_representations],
        bounds={"probe Spearman": (-1, 1), "width": (1, None)},
    )
    if saved_rows is not None:
        summary = pd.DataFrame(saved_rows).set_index("")
        summary.index.name = "representation"
        display(summary.sort_values("probe Spearman", ascending=False).round(4))
else:
    scores, predictions = {}, {}
    for name, X in representations.items():
        predictions[name], scores[name] = probe_on_split(X, y, train_rows, test_rows)
        print(f"  {name}: Spearman {scores[name]:.4f}")

    summary = pd.DataFrame({
        "probe Spearman": pd.Series(scores),
        "width": {k: v.shape[1] for k, v in representations.items()},
    })
    display(summary.sort_values("probe Spearman", ascending=False).round(4))


  Composition (GC, GC3, codon use, length): Spearman 0.2828
  NT v2 100M multi-species: Spearman 0.3124
  RNA-FM: Spearman 0.3083
  ESM-2 8M: Spearman 0.3092
  BLF three encoders concatenated: Spearman 0.3904
  LucaOne, gene stream: Spearman 0.4435
  LucaOne, protein stream: Spearman 0.4007
  LucaOne, both streams: Spearman 0.4828


,probe Spearman,width
"LucaOne, both streams",0.4828,5120
"LucaOne, gene stream",0.4435,2560
"LucaOne, protein stream",0.4007,2560
BLF three encoders concatenated,0.3904,1472
NT v2 100M multi-species,0.3124,512
ESM-2 8M,0.3092,320
RNA-FM,0.3083,640
"Composition (GC, GC3, codon use, length)",0.2828,87


**Reading the retained result.** The saved stability table places the two LucaOne vectors together above the concatenated BioLangFusion vectors under this ridge procedure. The [October 22 result record](../../docs/run-records.md#2026-10-22-stage4-human-probes-completed-half-life-translation-efficiency-and-lucaone-results) identifies that execution. This supports a predictive comparison of the selected representations; it does not establish that joint pretraining caused the gain or that the representation adds information conditionally beyond the calculated controls. The next cell estimates uncertainty in selected score differences.


### Compare score differences on the same test examples

The first cell displays the existing comparison CSV in saved-results mode, independently of the probe-score CSV. The following calculation cell is skipped in that mode, so no bootstrap is run and neither file is rewritten.

In computation mode, the paired bootstrap resamples test sequence groups and scores every predictor on the same resample. Identical sequences in the test partition stay together. Each row reports the first predictor's Spearman correlation minus the second's; the interval describes variation in that difference under this resampling procedure.

The “best BLF single encoder” is the encoder with the highest **observed test score** in this run. That selection is not repeated within the bootstrap, so its interval is conditional on the selected comparison. Although the helper is named `seed_averaged_bootstrap`, this notebook supplies one prediction array per representation: it does not repeat fits or partition assignments. `PROBE_SEEDS` does not change that.

Grouping repeated test sequences does not remove the known sequence overlap between training and test partitions. These intervals therefore do not establish performance on a strictly sequence-disjoint split. `first_wins` is the fraction of bootstrap draws with a positive difference, not the probability that a scientific hypothesis is true.


In [18]:
if READ_SAVED_RESULTS:
    # This independent saved table can be read even if the probe-score CSV is missing.
    saved_comparisons = saved_results.read_historical_csv(
        RESULTS_DIR / f"{BENCHMARK}_comparisons.csv",
        required=("comparison", "difference", "ci_low", "ci_high", "first_wins", "rows", "groups"),
        key=("comparison",), numeric=("difference", "ci_low", "ci_high", "first_wins"),
        integers=("rows", "groups"),
        expected_keys=[
            ("LucaOne both streams minus the best BLF single encoder",),
            ("LucaOne both streams minus the three concatenated",),
            ("LucaOne gene stream minus composition",),
        ],
        bounds={
            "difference": (-2, 2), "ci_low": (-2, 2), "ci_high": (-2, 2),
            "first_wins": (0, 1), "rows": (1, None), "groups": (1, None),
        },
    )
    if saved_comparisons is not None:
        if any(row["ci_low"] > row["ci_high"] or row["groups"] > row["rows"]
               for row in saved_comparisons):
            print("Saved comparisons have inconsistent interval/count fields; display skipped.")
        else:
            display(pd.DataFrame(saved_comparisons).set_index("comparison").round(4))


,difference,ci_low,ci_high,first_wins,rows,groups
comparison,,,,,,
LucaOne both streams minus the best BLF single encoder,0.1704,0.1370,0.2037,1.0,3579,3021
LucaOne both streams minus the three concatenated,0.0924,0.0658,0.1196,1.0,3579,3021
LucaOne gene stream minus composition,0.1606,0.1302,0.1908,1.0,3579,3021


In [ ]:
if not READ_SAVED_RESULTS:
    if "LucaOne, both streams" in predictions:
        comparisons = [
            (
                "LucaOne both streams minus the best BLF single encoder",
                "LucaOne, both streams",
                max(pooled, key=lambda k: scores[k]),  # Selected by observed test score.
            ),
            (
                "LucaOne both streams minus the three concatenated",
                "LucaOne, both streams",
                "BLF three encoders concatenated",
            ),
            (
                "LucaOne gene stream minus composition",
                "LucaOne, gene stream",
                "Composition (GC, GC3, codon use, length)",
            ),
        ]
        found = train.seed_averaged_bootstrap(
            "regression",
            y[test_rows],
            {k: [v] for k, v in predictions.items()},
            comparisons,
            groups=rows["sequence"].to_numpy()[test_rows],
            draws=BOOTSTRAP_DRAWS,
        )
        display(pd.DataFrame(found).set_index("comparison").round(4))

        # Publish both result tables after the paired comparisons are calculated.
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(found).to_csv(
            RESULTS_DIR / f"{BENCHMARK}_comparisons.csv", index=False,
        )
        summary.to_csv(RESULTS_DIR / f"{BENCHMARK}_probe_scores.csv")
    else:
        print("Run the LucaOne cells first; this cell compares what they produce.")


**Interpretation.** The retained comparison of concatenated representations has a positive interval for LucaOne minus BioLangFusion. That supports a difference under this fixed-split ridge comparison and its grouped test resampling. It does not by itself show that concatenation adds value over the strongest individual LucaOne stream, resolve the training/test overlap, or establish robustness across new splits. Those are separate comparisons. The following Phase A table changes the predictor as well as its representation handling, so it supplies context rather than another row in this matched probe comparison.


### Context from Phase A's trained token-level heads

Phase A trains a predictor over token representations, including BioLangFusion's TextCNN, whereas the table above uses ridge regression on pooled vectors. The architectures and fitting procedures differ; calling one categorically stronger would go beyond this comparison.

The helper below defines the contextual inventory; the following cell displays it. The inventory selects the full default Phase A training recipe with Equation 1 convolution, 100 channels and seed 0. It displays each variant and entropy penalty separately instead of averaging different penalties or seed counts. Duplicate identities stop the table; excluded configurations are shown. Stored scores are Spearman correlations for regression and accuracy proportions for classification. The inventory does not select a new test-set winner or establish full historical input identity, and it is not another matched ridge comparison. The retained output comes from the earlier reading procedure.


In [ ]:
def display_phase_a_context(benchmark, result_directory):
    """Display individual Phase A records without averaging or choosing a winner."""
    # Keep the Equation 1 seed-0 recipe explicit; do not average different MIL penalties.
    phase_a_report = swap_results.read_results(
        [result_directory],
        benchmark=benchmark,
        config=vars(train.TrainConfig(tconv="equation", seed=0)),
        encoders=dict(blf.ENCODER_OF_MODALITY),
    )
    if phase_a_report.issues:
        display(pd.DataFrame(phase_a_report.issues))
    phase_a = [
        {
            "variant": record["data"]["variant"],
            "entropy lambda": record["data"]["config"]["entropy_lambda"],
            "seed": record["data"]["config"]["seed"],
            "test score": record["data"]["test_score"],
        }
        for record in phase_a_report.require_unique()
    ]
    if phase_a:
        context_scores = pd.DataFrame(phase_a).set_index(["variant", "entropy lambda", "seed"])
        display(context_scores.sort_index().round(4))
        print(
            "Phase A trained token-level heads, seed 0: different predictor from the ridge "
            "comparison. Scores retain their stored units; no settings are averaged here."
        )
    else:
        print("No compatible Phase A seed-0 records; inspect the exclusions above.")


In [19]:
if READ_SAVED_RESULTS:
    # An ambiguous context table must not stop the independent GTEx reader.
    try:
        display_phase_a_context(BENCHMARK, STAGE4_DIR / "results" / "phaseA" / BENCHMARK)
    except ValueError as error:
        print(f"Phase A context unavailable: {error}; continuing to IsoFormer coverage.")
else:
    display_phase_a_context(BENCHMARK, STAGE4_DIR / "results" / "phaseA" / BENCHMARK)


,"Phase A, trained TextCNN"
variant,
concat,0.5582
cross,0.5527
unimodal:rna,0.5517
mil,0.5507
unimodal-grid:rna,0.5474
unimodal:dna,0.5391
unimodal:prot,0.5311


Different head, same rows and same split: not a matched comparison with the probe table, which is why it is in its own table.


### Focused follow-up: do the two LucaOne streams complement each other?

The earlier contrasts compare LucaOne with other encoders. Here the comparison is within LucaOne: **gene + protein minus gene alone**, and **gene + protein minus protein alone**. Both references are declared in advance; neither is selected using its test score. The protein input is translated from the coding sequence, so a gain concerns what the representations make accessible to ridge regression, not an additional independent measurement.

Keep `READ_SAVED_RESULTS = True`. Set `RUN_LUCAONE_COMPARISON = True` only when ready to calculate this CPU follow-up. It loads the local benchmark and the two existing pooled caches, validates their ordered inputs, and stops if either cache is missing or incompatible. It never fills a cache or loads LucaOne. With the switch off, the cells only read an existing follow-up report or state that it is absent.

The supplied training, validation and test partitions remain fixed. For each representation, the scaler is fitted on training rows and the ridge penalty is selected by validation mean squared error over `LUCAONE_ALPHAS`. The selected pipeline is then refitted on training plus validation and scored once on test. This differs from the historical internal `RidgeCV` calculation, so it receives its own result identity and does not replace the earlier scores. Initialization repeats are unnecessary for this deterministic fit. Identical sequences occur across the supplied partitions; the recorded overlap remains a limitation, not something this new tuning procedure fixes.


In [ ]:
lucaone_followup_inputs = None
if RUN_LUCAONE_COMPARISON:
    # Load local rows only: this loader has no download or model-inference path.
    followup_rows = datasets.blf_benchmark_rows(BENCHMARK)
    followup_sequences = followup_rows["sequence"].tolist()
    followup_texts = {
        "gene": followup_sequences,
        "prot": [tokens.grid_protein(s) for s in followup_sequences],
    }
    followup_cache_paths = {
        stream: LUCAONE_DIR / f"{CHECKPOINT}_{stream}.npz"
        for stream in followup_texts
    }
    followup_streams = {
        stream: lucaone.read_cached(
            texts, stream, followup_cache_paths[stream], checkpoint=CHECKPOINT,
        )
        for stream, texts in followup_texts.items()
    }
    followup_partition = followup_rows["Split"].to_numpy()
    if set(followup_partition) != {"train", "val", "test"}:
        raise ValueError("The focused LucaOne comparison requires train, val and test labels.")

    # Source row IDs stay distinct even when the sequence strings repeat.
    followup_row_ids = [f"{BENCHMARK}:{index}" for index in followup_rows.index]
    followup_data_path = datasets.DEFAULT_DATA_DIR / datasets.DATASETS[BENCHMARK].path
    followup_sources = {
        "dataset": followup_data_path,
        **{f"cache_{stream}": path for stream, path in followup_cache_paths.items()},
        "lucaone_reader": CODE_DIR / "shared_code" / "lucaone.py",
        "dataset_preparation": CODE_DIR / "shared_code" / "datasets.py",
    }
    lucaone_followup_inputs = {
        "name": f"{BENCHMARK}: LucaOne constituent comparison v1",
        "matrices": {
            "gene": followup_streams["gene"],
            "protein": followup_streams["prot"],
            "gene + protein": np.hstack([followup_streams["gene"], followup_streams["prot"]]),
        },
        "y": followup_rows["Value"].to_numpy(dtype=float),
        "row_ids": followup_row_ids,
        "groups": followup_sequences,
        "splits": {
            "published": tuple(np.flatnonzero(followup_partition == label)
                               for label in ("train", "val", "test")),
        },
        "contrasts": [
            ("combined minus gene", "gene + protein", "gene"),
            ("combined minus protein", "gene + protein", "protein"),
        ],
        "evidence": {
            "notebook_source": matched_probes.notebook_source(
                CODE_DIR / "stage1" / "lucaone_representation_comparison.ipynb",
                ("lucaone-complementarity-inputs", "lucaone-complementarity-results"),
            ),
            "sources": {name: {"path": str(path), "sha256": tokens.file_fingerprint(path)}
                        for name, path in followup_sources.items()},
            "cache_scope": "Ordered-input/checkpoint/stream/context metadata verified; historical revision and producer are not reconstructed.",
            "selection": "Fixed constituent contrasts; alpha by validation MSE; refit train+validation.",
            "generalization_limit": "Published partitions can share sequences; group bootstrap does not remove training/test overlap.",
        },
    }
else:
    print("LucaOne follow-up calculation is off; no datasets or embedding caches opened here.")


**Calculate or read the paired comparison.** The bootstrap resamples test-sequence groups jointly for both models. Its interval describes test-sample uncertainty conditional on the fitted models and supplied split; it does not measure transfer to unseen sequences or training-seed variation. The saved JSON retains actual test row IDs, targets, predictions, selected penalties, source hashes and overlap diagnostics. A matching completed analysis is reused before fitting, while a changed identity stops without overwriting it. Reading an existing JSON alone does not recheck today's cache contents.


In [ ]:
lucaone_followup_report = None
if RUN_LUCAONE_COMPARISON:
    if lucaone_followup_inputs is None:
        raise RuntimeError("Prepare the focused LucaOne inputs before calculating the report.")
    lucaone_followup_report, reused = matched_probes.run(
        LUCAONE_COMPARISON_PATH,
        **lucaone_followup_inputs,
        metric="spearman",
        alphas=LUCAONE_ALPHAS,
        draws=BOOTSTRAP_DRAWS,
        seed=0,  # Bootstrap sampling only; ridge fitting has no initialization seed.
        require_disjoint_groups=False,  # Retain and report the published-split overlap.
    )
    print("Reused completed analysis" if reused else "Saved new CPU comparison", LUCAONE_COMPARISON_PATH)
elif LUCAONE_COMPARISON_PATH.is_file():
    lucaone_followup_report = matched_probes.read(LUCAONE_COMPARISON_PATH)
else:
    print("Focused LucaOne comparison has not been saved; historical tables above remain available.")

if lucaone_followup_report is not None:
    display(pd.DataFrame(lucaone_followup_report["scores"]))
    display(pd.DataFrame(lucaone_followup_report["comparisons"]))


A positive combined-minus-gene difference would support an added contribution from the protein representation under this predictor and evaluation; a positive combined-minus-protein difference would support an added contribution from the gene representation. An interval spanning zero leaves that particular gain unresolved. These are exploratory follow-ups motivated by the earlier results, not evidence that the same benefit transfers to another organism, task or split.


---

## Findings, limitations and connection to the next comparison

The retained LucaOne results support the representation question: its separately obtained gene and protein vectors, concatenated and read by ridge regression, outperform the concatenated BioLangFusion vectors in the saved stability comparison. That finding belongs to the selected checkpoints, preprocessing, supplied split and predictor. It does not demonstrate that joint pretraining caused the improvement, remove known training/test sequence overlap, or establish performance for a different prediction head.

The input-length analysis explains the chosen scope. Separate streams retain the stability inputs under the selected context rule, while this notebook does not evaluate LucaOne on GTEx. Pooling is the local comparison choice; LucaOne does produce token representations. A token-level fusion adapter or windowed GTEx comparison would be a different experiment.

LucaOne and the [GTEx pilot](../stage3/gtex_frozen_fusion_reference.ipynb) contribute different evidence to the fusion question. LucaOne compares representation choices under a common pooled predictor; the GTEx pilot examines modality combinations with a local token-level cross-attention predictor. Their scores cannot establish that one model family or fusion method is better than the other, because the target, inputs and predictor change together.

The current context checks mark noncoding protein/joint lengths as inapplicable and keep Phase A configurations separate. Their historical saved outputs were preserved during the source repair. Any further comparison needs a stated question and compatible evidence; fixing a reader neither produces new evidence nor resolves missing historical provenance.
